1. Import libraries & setup

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml.feature import VectorAssembler, MinMaxScaler
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [0]:

dbutils.widgets.text("StorageName","")
StorageName = dbutils.widgets.get("StorageName")

dbutils.widgets.text("ContainerName","")
ContainerName = dbutils.widgets.get("ContainerName")

In [0]:

bronze_path = 'bronze'
common_options = {"header" :"true","inferSchema":"true"}

2. Load data

In [0]:
# load data to dataframe and display
file_path = f"abfss://{ContainerName}@{StorageName}.dfs.core.windows.net/{bronze_path}/Microsoft lab 3.csv"
df = (
    spark.read
    .option("header", "true")        # read first row as column names
    .option("inferSchema", "true")   # optional: infer data types
    .csv(file_path)
)

In [0]:
# Add a new column "diabetes_status" based on hba1c_percent thresholds
labeledData = df.withColumn(
    "diabetes_status",
    when(df.hba1c_percent > 6.5, "diabetic")
    .when((df.hba1c_percent > 5.7) & (df.hba1c_percent <= 6.5), "prediabetic")
    .otherwise("normal")
)

# Encode the label column into numeric form
indexer_label = StringIndexer(inputCol="diabetes_status", outputCol="label")
finalData = indexer_label.fit(labeledData).transform(labeledData).drop("hba1c_percent", "diabetes_status", "patient_id", "height_m", "weight_kg")

display(finalData.limit(5))

cholesterol_total,stabilized_glucose,hdl_cholesterol,chol_hdl_ratio,location,age,gender,body_frame,systolic_bp_1,diastolic_bp_1,minutes_post_meal,BMI,waist_hip_ratio,label
203,82,56,3.6,Buckingham,46,female,medium,118,59,720,22.130944261888523,0.7631578947368421,0.0
165,97,24,6.9,Buckingham,29,female,large,112,68,360,37.41920003371257,0.9583333333333334,0.0
228,92,37,6.2,Buckingham,58,female,large,190,92,180,48.37024068028249,0.8596491228070176,0.0
78,93,12,6.5,Buckingham,67,male,large,110,50,480,18.637828409539644,0.868421052631579,0.0
249,90,28,8.9,Buckingham,64,male,medium,138,80,300,27.824746566448155,1.0731707317073171,1.0


In [0]:
# Split the data
splits = finalData.randomSplit([0.7, 0.3])
train = splits[0]
test = splits[1]
print ("Training Rows:", train.count(), " Testing Rows:", test.count())

Training Rows: 275  Testing Rows: 112


In [0]:
display(train.limit(5))

patient_id,cholesterol_total,stabilized_glucose,hdl_cholesterol,chol_hdl_ratio,hba1c_percent,location,age,gender,body_frame,systolic_bp_1,diastolic_bp_1,minutes_post_meal,height_m,weight_kg,BMI,waist_hip_ratio
1001,165,97,24,6.9,4.44,Buckingham,29,female,large,112,68,360,1.6256,98.883056,37.41920003371257,0.9583333333333334
1005,249,90,28,8.9,7.72,Buckingham,64,male,medium,138,80,300,1.7271999999999998,83.007336,27.824746566448155,1.0731707317073171
1022,263,89,40,6.6,5.78,Buckingham,55,female,small,108,72,240,1.6001999999999998,91.625584,35.78229761649748,0.9
1030,238,75,36,6.6,4.47,Louisa,27,female,medium,130,80,720,1.524,77.11064,33.20048084540613,0.8536585365853658
1035,191,76,30,6.4,4.67,Louisa,36,male,medium,100,66,225,1.7526,83.007336,27.02407648041509,0.9


3. Encode the location, gender and body frame categorical column values as numeric

In [0]:
# Define indexers for each categorical column
indexer_location = StringIndexer(inputCol="location", outputCol="locationIdx")
indexer_gender = StringIndexer(inputCol="gender", outputCol="genderIdx")
indexer_body_frame = StringIndexer(inputCol="body_frame", outputCol="bodyFrameIdx")

# Chain them together in a pipeline
pipeline = Pipeline(stages=[indexer_location, indexer_gender, indexer_body_frame])

# Fit and transform the training data
indexedData = (pipeline.fit(train).transform(train).drop("location", "gender", "body_frame"))

display(indexedData.limit(5))

cholesterol_total,stabilized_glucose,hdl_cholesterol,chol_hdl_ratio,age,systolic_bp_1,diastolic_bp_1,minutes_post_meal,BMI,waist_hip_ratio,label,locationIdx,genderIdx,bodyFrameIdx
78,93,12,6.5,67,110,50,480,18.637828409539644,0.868421052631579,0.0,0.0,1.0,1.0
122,82,43,2.8,36,110,80,90,25.523036723518402,0.9111111111111111,0.0,1.0,0.0,0.0
129,110,42,3.1,56,140,75,90,19.387037970569732,0.8947368421052632,2.0,0.0,1.0,2.0
132,99,34,3.9,21,112,62,180,28.122760245520492,0.9069767441860465,0.0,0.0,0.0,1.0
134,105,42,3.2,48,178,120,240,24.822640420791046,0.9,0.0,0.0,1.0,1.0


3. Scaling numeric features 

Numeric values often exist on different ranges. During model training, the absolute units of measurement matter less than the relative differences between observations. If one feature has much larger values, it can dominate the learning process and distort the model’s predictions. To prevent this imbalance, numeric features are typically scaled to a common range — for example, mapping values to decimals between 0.0 and 1.0.

In [0]:
# Create a vector column containing all numeric features
numericFeatures = [c for c in indexedData.columns if c not in ["hba1c_percent"]]
numericColVector = VectorAssembler(inputCols=numericFeatures, outputCol="numericFeatures")
vectorizedData = numericColVector.transform(indexedData)


In [0]:
# Use a MinMax scaler to normalize the numeric values in the vector
minMax = MinMaxScaler(inputCol = numericColVector.getOutputCol(), outputCol="normalizedFeatures")
scaledData = minMax.fit(vectorizedData).transform(vectorizedData)

In [0]:
# Display the data with numeric feature vectors (before and after scaling) 
display(scaledData.select("numericFeatures", "normalizedFeatures").limit(5))

numericFeatures,normalizedFeatures
"Map(vectorType -> dense, length -> 14, values -> List(78.0, 93.0, 12.0, 6.5, 67.0, 110.0, 50.0, 480.0, 18.637828409539644, 0.868421052631579, 0.0, 0.0, 1.0, 1.0))","Map(vectorType -> dense, length -> 14, values -> List(0.0, 0.1393188854489164, 0.0, 0.5494505494505495, 0.6575342465753424, 0.07894736842105263, 0.0, 0.3310104529616725, 0.06621910241429305, 0.40768202260848435, 0.0, 0.0, 1.0, 0.5))"
"Map(vectorType -> dense, length -> 14, values -> List(122.0, 82.0, 43.0, 2.8, 36.0, 110.0, 80.0, 90.0, 25.523036723518402, 0.9111111111111111, 0.0, 1.0, 0.0, 0.0))","Map(vectorType -> dense, length -> 14, values -> List(0.16356877323420074, 0.10526315789473685, 0.28703703703703703, 0.14285714285714282, 0.2328767123287671, 0.07894736842105263, 0.40540540540540543, 0.059233449477351915, 0.23929253936502975, 0.5009494482935591, 0.0, 1.0, 0.0, 0.0))"
"Map(vectorType -> dense, length -> 14, values -> List(129.0, 110.0, 42.0, 3.1, 56.0, 140.0, 75.0, 90.0, 19.387037970569732, 0.8947368421052632, 2.0, 0.0, 1.0, 2.0))","Map(vectorType -> dense, length -> 14, values -> List(0.1895910780669145, 0.19195046439628485, 0.2777777777777778, 0.17582417582417584, 0.5068493150684932, 0.2763157894736842, 0.33783783783783783, 0.059233449477351915, 0.0850519783144412, 0.4651756411814757, 1.0, 0.0, 1.0, 1.0))"
"Map(vectorType -> dense, length -> 14, values -> List(132.0, 99.0, 34.0, 3.9, 21.0, 112.0, 62.0, 180.0, 28.122760245520492, 0.9069767441860465, 0.0, 0.0, 0.0, 1.0))","Map(vectorType -> dense, length -> 14, values -> List(0.2007434944237918, 0.15789473684210528, 0.2037037037037037, 0.26373626373626374, 0.0273972602739726, 0.09210526315789473, 0.16216216216216217, 0.12195121951219512, 0.30464177275874926, 0.49191685912240174, 0.0, 0.0, 0.0, 0.5))"
"Map(vectorType -> dense, length -> 14, values -> List(134.0, 105.0, 42.0, 3.2, 48.0, 178.0, 120.0, 240.0, 24.822640420791046, 0.9, 0.0, 0.0, 1.0, 1.0))","Map(vectorType -> dense, length -> 14, values -> List(0.20817843866171004, 0.17647058823529413, 0.2777777777777778, 0.18681318681318684, 0.3972602739726027, 0.5263157894736842, 0.945945945945946, 0.16376306620209058, 0.2216866820237923, 0.47667436489607395, 0.0, 0.0, 1.0, 0.5))"


In [0]:
#Prepare feature and labels for training
featVect = VectorAssembler(inputCols=["label", "normalizedFeatures"], outputCol="featuresVector")
preppedData = featVect.transform(scaledData)[col("featuresVector").alias("features"), col("label").alias("label")]
display(preppedData)


features,label
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.0, 0.1393188854489164, 0.0, 0.5494505494505495, 0.6575342465753424, 0.07894736842105263, 0.0, 0.3310104529616725, 0.06621910241429305, 0.40768202260848435, 0.0, 0.0, 1.0, 0.5))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.16356877323420074, 0.10526315789473685, 0.28703703703703703, 0.14285714285714282, 0.2328767123287671, 0.07894736842105263, 0.40540540540540543, 0.059233449477351915, 0.23929253936502975, 0.5009494482935591, 0.0, 1.0, 0.0, 0.0))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(2.0, 0.1895910780669145, 0.19195046439628485, 0.2777777777777778, 0.17582417582417584, 0.5068493150684932, 0.2763157894736842, 0.33783783783783783, 0.059233449477351915, 0.0850519783144412, 0.4651756411814757, 1.0, 0.0, 1.0, 1.0))",2.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.2007434944237918, 0.15789473684210528, 0.2037037037037037, 0.26373626373626374, 0.0273972602739726, 0.09210526315789473, 0.16216216216216217, 0.12195121951219512, 0.30464177275874926, 0.49191685912240174, 0.0, 0.0, 0.0, 0.5))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.20817843866171004, 0.17647058823529413, 0.2777777777777778, 0.18681318681318684, 0.3972602739726027, 0.5263157894736842, 0.945945945945946, 0.16376306620209058, 0.2216866820237923, 0.47667436489607395, 0.0, 0.0, 1.0, 0.5))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.21189591078066913, 0.12383900928792571, 0.2037037037037037, 0.27472527472527475, 0.136986301369863, 0.13157894736842105, 0.14864864864864866, 0.16376306620209058, 0.11222523093565379, 0.0456276137569442, 0.0, 0.0, 0.0, 1.0))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.21189591078066913, 0.12383900928792571, 0.32407407407407407, 0.15384615384615383, 0.0273972602739726, 0.07894736842105263, 0.24324324324324326, 0.003484320557491289, 0.1730869364833076, 0.24699472967371347, 0.0, 1.0, 1.0, 1.0))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.22304832713754646, 0.1455108359133127, 0.25925925925925924, 0.21978021978021978, 0.2602739726027397, 0.2763157894736842, 0.5405405405405406, 0.2264808362369338, 0.27518670430938036, 0.24699472967371347, 0.0, 1.0, 0.0, 1.0))",0.0
"Map(vectorType -> dense, length -> 15, values -> List(1.0, 0.2379182156133829, 0.33126934984520123, 0.12037037037037036, 0.46153846153846156, 0.3561643835616438, 0.4407894736842105, 0.8783783783783784, 0.49825783972125437, 0.3549770882185402, 0.5427251732101617, 0.5, 0.0, 1.0, 0.5))",1.0
"Map(vectorType -> dense, length -> 15, values -> List(0.0, 0.241635687732342, 1.0, 0.31481481481481477, 0.17582417582417584, 0.6712328767123288, 0.2631578947368421, 0.43243243243243246, 0.059233449477351915, 0.2197607824577836, 0.3903002309468822, 0.0, 1.0, 1.0, 1.0))",0.0


## Conclusion

The dataset has been fully prepared for machine learning.  
- Categorical fields were indexed.  
- Numeric features were normalized.  
- A diabetes status label was derived from `hba1c_percent`.  
- All predictors were assembled into a single feature vector.  

We now have a clean structure of **features** and **labels**, ready to move into the machine learning phase for training and evaluation.
